# CMO emitter-aware Wikipedia airborne-radar Neo4j KG

Replacement for the previous `wikipedia_airborne_radars_neo4j_kg.ipynb`. It wipes the selected Neo4j database and rebuilds a reference KG with an ontology aligned to `LuaHistory_2026-06-23.txt`: emitter aliases/types, platform identity and variants, operator country, kinematics, and location/geography.

In [1]:
from pathlib import Path
import os, sys, json

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'combat_id_calibration').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from combat_id_calibration.graph_ingest import load_documents, extract_facts, write_facts_jsonl, load_facts_jsonl, populate_neo4j, _validate_neo4j_credentials

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = 'password123' #os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'qwen3.5:9b')
OLLAMA_URL = os.getenv('OLLAMA_URL', 'http://localhost:11434')
FACTS_JSONL = REPO_ROOT / 'wikipedia_emitter_ontology_facts.jsonl'
print(REPO_ROOT)

C:\Users\theon\CMO-Sensor-Fusion


## Ontology

Static sources are mined for facts that can meet dynamic CMO observations. Important predicates include `HAS_SENSOR`, `USES_RADAR`, `HAS_EMITTER`, `EMITTER_TYPE`, `PLATFORM_TYPE`, `VARIANT_OF`, `HAS_VARIANT`, `OPERATED_BY`, `OPERATOR_COUNTRY`, `TYPICAL_SPEED_KT`, `MAX_SPEED_KT`, `SERVICE_CEILING_M`, `BASED_AT`, `DEPLOYED_TO`, and `OPERATES_IN`. `graph_ingest.py` writes the auditable generic `FACT` edge and also materializes selected typed relationships/labels (`Platform`, `Sensor`, `Operator`, `Country`, `Location`) so hypothesis queries can traverse the reference graph directly.

In [2]:
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = 'password123' #os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None

def _validate_neo4j_credentials(user: str | None, password: str | None) -> tuple[str, str]:
    """Validate Neo4j credentials before building the driver auth token."""

    username = "" if user is None else str(user).strip()
    if not username:
        raise ValueError("Neo4j username is required; set --neo4j-user or NEO4J_USER before ingestion.")
    if password is None or not str(password):
        raise ValueError(
            "Neo4j password is required; set --neo4j-password or NEO4J_PASSWORD before ingestion. "
            "In the notebook, replace the placeholder empty string with your Neo4j password."
        )
    return username, str(password)

def wipe_neo4j(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD, database=NEO4J_DATABASE):
    user, password = _validate_neo4j_credentials(user, password)
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        driver.verify_connectivity()
        with driver.session(**({'database': database} if database else {})) as session:
            session.run('MATCH (n) DETACH DELETE n')
    finally:
        driver.close()

wipe_neo4j()
print('Neo4j graph wiped')

Neo4j graph wiped


In [3]:
WIKIPEDIA_URLS = [
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29',
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-35',
    'https://en.wikipedia.org/wiki/Zhuk_(radar)',
    'https://en.wikipedia.org/wiki/N010_Zhuk',
    'https://en.wikipedia.org/wiki/Ukrainian_Air_Force',
    'https://en.wikipedia.org/wiki/Russian_Air_Force',
    'https://en.wikipedia.org/wiki/Eurofighter_Typhoon',
    'https://en.wikipedia.org/wiki/CAPTOR-E',
    'https://en.wikipedia.org/wiki/Sukhoi_Su-27',
    'https://en.wikipedia.org/wiki/McDonnell_Douglas_F-15_Eagle',
    'https://en.wikipedia.org/wiki/Eurofighter_Typhoon',
    'https://en.wikipedia.org/wiki/R-77',
    'https://en.wikipedia.org/wiki/Euroradar_CAPTOR',
    'https://en.wikipedia.org/wiki/General_Dynamics_F-16_Fighting_Falcon',
    'https://en.wikipedia.org/wiki/Mech_radar',
    'https://en.wikipedia.org/wiki/AN/APG-68',
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29K#Operators',
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29M#Operators',
    'https://en.wikipedia.org/wiki/List_of_Mikoyan_MiG-29_operators',
    'https://en.wikipedia.org/wiki/General_Dynamics_F-16_Fighting_Falcon',
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29K'
]
PDF_PATHS = []

In [7]:
# Wipe the selected Neo4j graph before rebuilding.
def wipe_neo4j(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD, database=NEO4J_DATABASE):
    user, password = _validate_neo4j_credentials(user, password)
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        driver.verify_connectivity()
        with driver.session(**({'database': database} if database else {})) as session:
            session.run('MATCH (n) DETACH DELETE n')
    finally:
        driver.close()

wipe_neo4j()
print('Neo4j graph wiped')

Neo4j graph wiped


In [ ]:
documents = load_documents(PDF_PATHS, WIKIPEDIA_URLS)
print(f'Loaded {len(documents)} documents')
facts = extract_facts(documents, model=OLLAMA_MODEL, ollama_url=OLLAMA_URL, max_chars=5000, overlap=500)

FACTS_JSONL = REPO_ROOT / 'wikipedia_emitter_ontology_facts_03.jsonl'
write_facts_jsonl(facts, FACTS_JSONL)
print(f'Wrote {len(facts)} extracted facts to {FACTS_JSONL}')

https://en.wikipedia.org/wiki/Mikoyan_MiG-29
https://en.wikipedia.org/wiki/Mikoyan_MiG-35
https://en.wikipedia.org/wiki/Zhuk_(radar)
https://en.wikipedia.org/wiki/N010_Zhuk
https://en.wikipedia.org/wiki/Ukrainian_Air_Force
https://en.wikipedia.org/wiki/Russian_Air_Force
https://en.wikipedia.org/wiki/Eurofighter_Typhoon
https://en.wikipedia.org/wiki/CAPTOR-E
https://en.wikipedia.org/wiki/Sukhoi_Su-27
https://en.wikipedia.org/wiki/McDonnell_Douglas_F-15_Eagle
https://en.wikipedia.org/wiki/Eurofighter_Typhoon
https://en.wikipedia.org/wiki/R-77
https://en.wikipedia.org/wiki/Euroradar_CAPTOR
https://en.wikipedia.org/wiki/General_Dynamics_F-16_Fighting_Falcon
https://en.wikipedia.org/wiki/Mech_radar
https://en.wikipedia.org/wiki/AN/APG-68
https://en.wikipedia.org/wiki/Mikoyan_MiG-29K#Operators
https://en.wikipedia.org/wiki/Mikoyan_MiG-29M#Operators
https://en.wikipedia.org/wiki/List_of_Mikoyan_MiG-29_operators
https://en.wikipedia.org/wiki/General_Dynamics_F-16_Fighting_Falcon
https://en.wik

[graph-ingest] starting fact extraction with model='qwen3.5:9b', ollama_url='http://localhost:11434', max_chars=5000, overlap=500
[graph-ingest] document 1: title='Mikoyan MiG-29', source_id=61eb8344713ba93e, type=wikipedia, text_chars=177160, chunks=39
[graph-ingest] document 1 chunk 1/39: sending 5000 chars to Ollama
[graph-ingest] document 1 chunk 1/39: received 3198 chars; preview='{\\n  "facts": [\\n    {\\n      "subject": "Mikoyan MiG-29",\\n      "predicate": "NATO_REPORTING_NAME",\\n      "object": "Fulcrum",\\n      "evidence": "\\"NATO reporting name : Fulcurm\\"",\\n      "confidence": 1.0\\n    },\\n    {\\n      "subject": "Mikoyan MiG-29",\\n      "predicate": "OPERATED_BY",\\n      "object"'
[graph-ingest] document 1 chunk 1/39: JSON candidate 1 contains facts=14
[graph-ingest] document 1 chunk 1/39: normalized 14 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 1/39: added 14 fact(s)
[graph-ingest] document 1 chunk 2/39: sending 5000 chars to Ollama
[

In [4]:
FACTS_JSONL = REPO_ROOT / 'wikipedia_emitter_ontology_facts_03.jsonl'
facts = load_facts_jsonl(FACTS_JSONL)
populate_neo4j(facts, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
print('Populated emitter-aware reference KG')

Populated emitter-aware reference KG


In [5]:
# Smoke-test emitter/platform/operator coverage for the first LuaHistory emitter.
from combat_id_calibration.hypothesis_generation import graph_hypothesis_query, emitter_aliases
from neo4j import GraphDatabase
aliases = emitter_aliases('Slot Back [N-010 Zhuk-M]')
query, params = graph_hypothesis_query(aliases, limit=20)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
try:
    with driver.session(**({'database': NEO4J_DATABASE} if NEO4J_DATABASE else {})) as session:
        rows = [dict(r) for r in session.run(query, **params)]
finally:
    driver.close()
rows

[{'hypothesis': 'MiG-29KR',
  'operator_nation': 'Russia',
  'aircraft_variant': 'MiG-29KR',
  'emitter_variant': 'Zhuk',
  'matched_aliases': ['Zhuk'],
  'support_count': 430,
  'evidence_paths': [['MENTIONED_IN', 'MENTIONED_IN', 'IS_A', 'FACT'],
   ['MENTIONED_IN', 'MENTIONED_IN', 'FACT', 'IS_A'],
   ['MENTIONED_IN', 'MENTIONED_IN', 'HAS_MODE', 'FACT'],
   ['MENTIONED_IN', 'MENTIONED_IN', 'FACT', 'HAS_MODE'],
   ['VARIANT_OF', 'FACT', 'HAS_SENSOR'],
   ['FACT', 'OPERATOR_COUNTRY', 'FACT', 'HAS_SENSOR'],
   ['OPERATOR_COUNTRY', 'OPERATOR_COUNTRY', 'FACT', 'HAS_SENSOR'],
   ['FACT', 'OPERATED_BY', 'FACT', 'HAS_SENSOR'],
   ['OPERATED_BY', 'OPERATED_BY', 'FACT', 'HAS_SENSOR'],
   ['SERVICE_WITH', 'OPERATED_BY', 'FACT', 'HAS_SENSOR'],
   ['MENTIONED_IN', 'MENTIONED_IN', 'FACT', 'HAS_SENSOR'],
   ['FACT', 'FACT', 'FACT', 'HAS_SENSOR'],
   ['OPERATOR_COUNTRY', 'FACT', 'FACT', 'HAS_SENSOR'],
   ['OPERATED_BY', 'FACT', 'FACT', 'HAS_SENSOR'],
   ['SERVICE_WITH', 'FACT', 'FACT', 'HAS_SENSOR'],